# Prétraitement du Dataset FER2013
### Projet : Reconnaissance des Émotions Faciales — GL4 INSAT


## 0. Imports et configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array

# Reproductibilite
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Chemins (adapter si besoin)
TRAIN_DIR = 'data/train'
TEST_DIR  = 'data/test'

# Parametres images
IMG_SIZE   = (48, 48)
BATCH_SIZE = 64

# Labels (ordre alphabetique = ordre detecte par Keras)
EMOTION_LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
EMOTION_COLORS = ['#e74c3c','#8e44ad','#e67e22','#f1c40f','#95a5a6','#3498db','#1abc9c']

os.makedirs('results', exist_ok=True)
os.makedirs('data/preprocessed', exist_ok=True)

print('TensorFlow version :', tf.__version__)
print('GPU disponible     :', tf.config.list_physical_devices('GPU'))
print('Imports OK')

---
## 1. Exploration initiale du dataset

In [ ]:
def count_images(root_dir):
    """Compte le nombre d'images par sous-dossier (classe)."""
    counts = {}
    root = Path(root_dir)
    for class_dir in sorted(root.iterdir()):
        if class_dir.is_dir():
            n = len(list(class_dir.glob('*.jpg')) +
                    list(class_dir.glob('*.png')) +
                    list(class_dir.glob('*.jpeg')))
            counts[class_dir.name] = n
    return counts

train_counts = count_images(TRAIN_DIR)
test_counts  = count_images(TEST_DIR)

print('=== Distribution des images ===')
print(f'{"Classe":<12} {"Train":>8} {"Test":>8} {"Total":>8}')
print('-' * 42)
total_train = total_test = 0
for cls in EMOTION_LABELS:
    tr = train_counts.get(cls, 0)
    te = test_counts.get(cls, 0)
    total_train += tr
    total_test  += te
    print(f'{cls:<12} {tr:>8} {te:>8} {tr+te:>8}')
print('-' * 42)
print(f'{"TOTAL":<12} {total_train:>8} {total_test:>8} {total_train+total_test:>8}')

ratio = max(train_counts.values()) / min(train_counts.values())
print(f'\nRatio max/min (train) : {ratio:.1f}x  -> desequilibre significatif')

In [ ]:
# Visualisation de la distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (title, counts) in zip(axes, [('Train', train_counts), ('Test', test_counts)]):
    classes = [c for c in EMOTION_LABELS if c in counts]
    values  = [counts[c] for c in classes]
    bars = ax.bar(classes, values, color=EMOTION_COLORS, edgecolor='white', linewidth=0.8)
    ax.axhline(np.mean(values), color='red', linestyle='--', linewidth=1.5,
               label=f'Moyenne : {int(np.mean(values))}')
    ax.set_title(f'Distribution - {title}', fontsize=13, fontweight='bold')
    ax.set_ylabel("Nombre d'images")
    ax.tick_params(axis='x', rotation=30)
    ax.legend()
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('results/distribution_classes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Visualisation d'exemples bruts par classe
fig, axes = plt.subplots(2, 7, figsize=(16, 5))
fig.suptitle('Exemples bruts - 2 images par classe', fontsize=13, fontweight='bold')

for col, cls in enumerate(EMOTION_LABELS):
    class_path = Path(TRAIN_DIR) / cls
    images = list(class_path.glob('*.jpg')) + list(class_path.glob('*.png'))
    for row in range(2):
        img = load_img(images[row], color_mode='grayscale', target_size=IMG_SIZE)
        axes[row][col].imshow(img_to_array(img).squeeze(), cmap='gray')
        axes[row][col].axis('off')
        if row == 0:
            axes[row][col].set_title(cls, fontsize=10,
                                     color=EMOTION_COLORS[col], fontweight='bold')

plt.tight_layout()
plt.savefig('results/exemples_bruts.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Chargement avec ImageDataGenerator

In [ ]:
# Generateur TRAIN avec augmentation + split validation
train_datagen = ImageDataGenerator(
    rescale=1.0/255,           # normalisation [0, 1]
    validation_split=0.20,     # 20% du train -> validation
    rotation_range=15,
    horizontal_flip=True,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    shear_range=0.05,
    fill_mode='nearest'
)

# Generateur TEST/VAL : rescale uniquement, pas d'augmentation
test_datagen = ImageDataGenerator(rescale=1.0/255)

print('Generateurs definis OK')

In [ ]:
# Flux TRAIN
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='sparse',   # labels entiers 0-6
    subset='training',
    shuffle=True,
    seed=SEED
)

# Flux VALIDATION (sans augmentation)
val_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='validation',
    shuffle=False,
    seed=SEED
)

# Flux TEST
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    shuffle=False
)

# Mapping classe -> index
idx_to_class = {v: k for k, v in train_generator.class_indices.items()}
print('\nMapping classes :', train_generator.class_indices)

---
## 3. Verification de la normalisation

In [ ]:
batch_X, batch_y = next(train_generator)

print('=== Verification normalisation ===')
print(f'Shape du batch  : {batch_X.shape}')   # (64, 48, 48, 1)
print(f'Dtype           : {batch_X.dtype}')
print(f'Valeur min      : {batch_X.min():.4f}')
print(f'Valeur max      : {batch_X.max():.4f}')
print(f'Valeur moyenne  : {batch_X.mean():.4f}')
print(f'Labels presents : {np.unique(batch_y.astype(int))}')

assert batch_X.min() >= 0.0 and batch_X.max() <= 1.0
print('Normalisation [0, 1] correcte')

In [ ]:
# Comparaison visuelle avant / apres normalisation
sample_path = list((Path(TRAIN_DIR) / 'happy').glob('*.jpg'))[0]
img_raw  = img_to_array(load_img(sample_path, color_mode='grayscale', target_size=IMG_SIZE))
img_norm = img_raw / 255.0

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle('Effet de la normalisation', fontsize=12, fontweight='bold')

axes[0].imshow(img_raw.squeeze(), cmap='gray', vmin=0, vmax=255)
axes[0].set_title(f'Brut\nmin={img_raw.min():.0f} | max={img_raw.max():.0f}', fontsize=10)
axes[0].axis('off')

axes[1].imshow(img_norm.squeeze(), cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Normalise [0,1]\nmin={img_norm.min():.2f} | max={img_norm.max():.2f}', fontsize=10)
axes[1].axis('off')

axes[2].hist(img_norm.flatten(), bins=50, color='steelblue', edgecolor='white')
axes[2].set_title('Histogramme des pixels\n(apres normalisation)', fontsize=10)
axes[2].set_xlabel('Valeur pixel')
axes[2].set_ylabel('Frequence')

plt.tight_layout()
plt.savefig('results/normalisation.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Calcul des class_weights

In [ ]:
all_labels = train_generator.classes

class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_labels),
    y=all_labels
)
class_weight_dict = dict(enumerate(class_weights_array))

print('=== Class Weights ===')
for k, w in class_weight_dict.items():
    print(f'  Classe {k} - {idx_to_class[k]:10s} : {w:.4f}')

print('\nA passer a model.fit(class_weight=class_weight_dict)')

In [ ]:
classes_ordered  = [idx_to_class[i] for i in range(len(EMOTION_LABELS))]
counts_ordered   = [train_counts.get(c, 0) for c in classes_ordered]
weights_ordered  = [class_weight_dict[i] for i in range(len(EMOTION_LABELS))]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.bar(classes_ordered, counts_ordered, color=EMOTION_COLORS, edgecolor='white')
ax1.set_title("Nombre d'images par classe (train)", fontweight='bold')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=30)
for i, v in enumerate(counts_ordered):
    ax1.text(i, v + 30, str(v), ha='center', fontsize=8)

ax2.bar(classes_ordered, weights_ordered, color=EMOTION_COLORS, edgecolor='white')
ax2.axhline(1.0, color='red', linestyle='--', linewidth=1.2, label='Poids = 1')
ax2.set_title('Class Weights correspondants', fontweight='bold')
ax2.set_ylabel('Poids')
ax2.tick_params(axis='x', rotation=30)
ax2.legend()
for i, v in enumerate(weights_ordered):
    ax2.text(i, v + 0.02, f'{v:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('results/class_weights.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Verification du split Train / Validation

In [ ]:
total_split = train_generator.samples + val_generator.samples
print('=== Taille des splits ===')
print(f'  Train      : {train_generator.samples} images ({train_generator.samples/total_split*100:.0f}%)')
print(f'  Validation : {val_generator.samples} images ({val_generator.samples/total_split*100:.0f}%)')
print(f'  Test       : {test_generator.samples} images')
print(f'\n  Batches train : {len(train_generator)}')
print(f'  Batches val   : {len(val_generator)}')
print(f'  Batches test  : {len(test_generator)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, (gen, title) in zip(axes, [(train_generator, 'Train (80%)'),
                                     (val_generator,   'Validation (20%)')]):
    unique, cnts = np.unique(gen.classes, return_counts=True)
    pcts = cnts / cnts.sum() * 100
    ax.bar([idx_to_class[i] for i in unique], pcts, color=EMOTION_COLORS, edgecolor='white')
    ax.set_title(f'{title} - n={gen.samples}', fontweight='bold')
    ax.set_ylabel('%')
    ax.set_ylim(0, 30)
    ax.tick_params(axis='x', rotation=30)
    for i, p in enumerate(pcts):
        ax.text(i, p + 0.3, f'{p:.1f}%', ha='center', fontsize=8)

plt.suptitle('Distribution des classes - Train vs Validation', fontweight='bold')
plt.tight_layout()
plt.savefig('results/split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Distributions coherentes entre les splits')

---
## 6. Visualisation de la Data Augmentation

In [ ]:
# Generateur d'augmentation standalone pour visualisation
aug_datagen = ImageDataGenerator(
    rotation_range=15,
    horizontal_flip=True,
    width_shift_range=0.10,
    height_shift_range=0.10,
    zoom_range=0.10,
    shear_range=0.05,
    fill_mode='nearest'
)

sample_path = list((Path(TRAIN_DIR) / 'happy').glob('*.jpg'))[0]
sample_img  = img_to_array(load_img(sample_path, color_mode='grayscale', target_size=IMG_SIZE)) / 255.0
sample_4d   = sample_img.reshape(1, 48, 48, 1)
aug_iter    = aug_datagen.flow(sample_4d, batch_size=1, seed=SEED)

fig, axes = plt.subplots(3, 7, figsize=(16, 7))
fig.suptitle('Data Augmentation - 20 versions de la meme image (happy)', fontsize=12, fontweight='bold')

axes[0][0].imshow(sample_img.squeeze(), cmap='gray')
axes[0][0].set_title('Original', fontsize=9, color='green', fontweight='bold')
axes[0][0].axis('off')

count = 1
for row in range(3):
    for col in range(7):
        if row == 0 and col == 0:
            continue
        aug_img = next(aug_iter)[0].squeeze()
        axes[row][col].imshow(aug_img, cmap='gray')
        axes[row][col].set_title(f'Aug #{count}', fontsize=8)
        axes[row][col].axis('off')
        count += 1

plt.tight_layout()
plt.savefig('results/data_augmentation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Apercu d'un batch complet
batch_X, batch_y = next(train_generator)

fig, axes = plt.subplots(4, 8, figsize=(16, 8))
fig.suptitle(f'Apercu batch train (32 premieres images sur {BATCH_SIZE})', fontsize=12, fontweight='bold')

for i, ax in enumerate(axes.flatten()):
    ax.imshow(batch_X[i].squeeze(), cmap='gray')
    label = int(batch_y[i])
    ax.set_title(idx_to_class[label], fontsize=7, color=EMOTION_COLORS[label])
    ax.axis('off')

plt.tight_layout()
plt.savefig('results/batch_apercu.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Recapitulatif et sauvegarde

In [ ]:
print('=' * 55)
print('         RECAPITULATIF DU PIPELINE')
print('=' * 55)
print(f'\nDataset          : dossiers train/ et test/')
print(f'Format images    : grayscale, {IMG_SIZE[0]}x{IMG_SIZE[1]} px')
print(f'Normalisation    : [0, 1]  (rescale=1/255)')
print(f'Augmentation     : rotation+-15 deg, flip, zoom+-10%, shift+-10%')
print(f'Split            : 80% train / 20% val (validation_split=0.2)')
print(f'\n  Train      : {train_generator.samples} images - {len(train_generator)} batches')
print(f'  Validation : {val_generator.samples} images - {len(val_generator)} batches')
print(f'  Test       : {test_generator.samples} images - {len(test_generator)} batches')
print(f'\nClass Weights :')
for k, w in class_weight_dict.items():
    print(f'  {k} - {idx_to_class[k]:10s} : {w:.4f}')
print('\n' + '=' * 55)
print('  Pipeline pret pour la modelisation (CR3) !')
print('=' * 55)

In [ ]:
# Export data to .npy format for faster loading in training notebooks
print('Exporting data to .npy format...\n')

# Reset generators and extract all data
train_generator.reset()
val_generator.reset()
test_generator.reset()

# Get all data from generators
X_train_list, y_train_list = [], []
X_val_list, y_val_list = [], []
X_test_list, y_test_list = [], []

# Extract training data
for _ in range(len(train_generator)):
    X_batch, y_batch = next(train_generator)
    X_train_list.append(X_batch)
    y_train_list.append(np.argmax(y_batch, axis=1) if y_batch.ndim > 1 and y_batch.shape[1] > 1 else y_batch.astype(int))

# Extract validation data
for _ in range(len(val_generator)):
    X_batch, y_batch = next(val_generator)
    X_val_list.append(X_batch)
    y_val_list.append(np.argmax(y_batch, axis=1) if y_batch.ndim > 1 and y_batch.shape[1] > 1 else y_batch.astype(int))

# Extract test data
for _ in range(len(test_generator)):
    X_batch, y_batch = next(test_generator)
    X_test_list.append(X_batch)
    y_test_list.append(np.argmax(y_batch, axis=1) if y_batch.ndim > 1 and y_batch.shape[1] > 1 else y_batch.astype(int))

# Concatenate
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_val = np.concatenate(X_val_list, axis=0)
y_val = np.concatenate(y_val_list, axis=0)
X_test = np.concatenate(X_test_list, axis=0)
y_test = np.concatenate(y_test_list, axis=0)

# Save to .npy files
np.save('data/preprocessed/X_train.npy', X_train)
np.save('data/preprocessed/y_train.npy', y_train)
np.save('data/preprocessed/X_val.npy', X_val)
np.save('data/preprocessed/y_val.npy', y_val)
np.save('data/preprocessed/X_test.npy', X_test)
np.save('data/preprocessed/y_test.npy', y_test)

print(f'✓ X_train saved: {X_train.shape}')
print(f'✓ y_train saved: {y_train.shape}')
print(f'✓ X_val saved: {X_val.shape}')
print(f'✓ y_val saved: {y_val.shape}')
print(f'✓ X_test saved: {X_test.shape}')
print(f'✓ y_test saved: {y_test.shape}')

# Sauvegarde des metadonnees pour les notebooks suivants
with open('data/preprocessed/class_weights.json', 'w') as f:
    json.dump({str(k): v for k, v in class_weight_dict.items()}, f, indent=2)

with open('data/preprocessed/class_mapping.json', 'w') as f:
    json.dump(train_generator.class_indices, f, indent=2)

print('\nFichiers sauvegardes :')
print('  data/preprocessed/X_train.npy, X_val.npy, X_test.npy')
print('  data/preprocessed/y_train.npy, y_val.npy, y_test.npy')
print('  data/preprocessed/class_weights.json')
print('  data/preprocessed/class_mapping.json')